<a id="top"></a>

# Teach Open Biocatalysis

## WET-01a: Enzyme Kinetic Assay - Wet-Lab Protocol & Data Acquisition

> **You are here:** this is the **wet-lab module** of the *Teach Open Biocatalysis* series. Bring this notebook to the bench — it walks you through the experiment, captures your measurements, and prepares the dataset for the downstream modules (RDM-01a, KIN-01a).

---

### Course Goal

*Teach Open Biocatalysis* provides accessible, open-source educational materials that bridge classical enzyme kinetics with modern data management (FAIR) and computational modeling. By the end of the series you will be able to:

- Design and perform enzyme kinetic experiments at the bench
- Apply FAIR principles (Findable, Accessible, Interoperable, Reusable) to biocatalytic data
- Use computational tools for kinetic parameter determination
- Integrate wet-lab data into standardized formats (EnzymeML, STRENDA DB)

---

### Where this notebook fits in the workflow

```mermaid
flowchart LR
    A[WET-01a<br/>Wet-lab assay<br/>and data acquisition] --> B[RDM-01a<br/>EnzymeML and<br/>FAIR data management]
    B --> C[KIN-01a<br/>Kinetic modeling<br/>and parameter fitting]
    style A fill:#fde68a,stroke:#b45309,stroke-width:2px,color:#111
    style B fill:#e5e7eb,stroke:#374151,color:#111
    style C fill:#e5e7eb,stroke:#374151,color:#111
```

> If the diagram above does not render in your environment (e.g. plain Jupyter without a Mermaid extension), the workflow is simply: **WET-01a → RDM-01a → KIN-01a**. It renders natively on GitHub, JupyterLab ≥ 4.1, and VS Code.

---

### Table of Contents

- [0. Setup](#sec-setup)
- [Part 1 — Theoretical Background](#sec-part1)
  - [1.1 FDH-catalyzed NADH cofactor regeneration](#sec-1-1)
  - [1.2 The FDH reaction and spectrophotometric detection](#sec-1-2)
  - [1.3 Determination of reactant concentrations](#sec-1-3)
  - [1.4 Specific activity of FDH](#sec-1-4)
- [Part 2 — Experimental Setup and Protocols](#sec-part2)
  - [2.1 Hardware](#sec-2-1)
  - [2.2 Consumables](#sec-2-2)
  - [2.3 Buffers, solutions and conditions](#sec-2-3)
  - [2.4 Metadata and the STRENDA guidelines](#sec-2-4)
  - [2.5 Pipetting scheme and initial concentrations](#sec-2-5)
- [Part 3 — Data Acquisition](#sec-part3)
  - [3.1 Measurement protocol step by step](#sec-3-1)
  - [3.2 Entering your absorbance values](#sec-3-2)
  - [3.3 Loading the measurements (with fallback)](#sec-3-3)
- [Part 4 — Data Processing and Analysis](#sec-part4)
  - [4.1 Theoretical background for the calculations](#sec-4-1)
  - [4.2 Calculation of concentrations and specific activity](#sec-4-2)
- [Part 5 — FAIR Data Export and Module Integration](#sec-part5)
- [Conclusion & Outlook](#sec-outlook)
- [References & Resources](#sec-references)
- [Acknowledgments](#sec-ack)

---

### Authors & Contributors

- **Niklas-Maximilian Epping**, Institute of Technical Biocatalysis, Hamburg University of Technology — niklas.epping@tuhh.de
- **Andreas Liese**, Institute of Technical Biocatalysis, Hamburg University of Technology — liese@tuhh.de

Built on community standards (STRENDA, EnzymeML). Special thanks to **Nicole Klassen** and **Levin Wagner** for generating reference data and helping to re-design the wet-lab course content from the perspective of technical-assistants-in-training — their feedback significantly shaped this material.

---

### What you will do in this notebook

| Step | What you do | Why it matters |
|------|-------------|----------------|
| 1 | Read the theory (FDH reaction, Lambert–Beer, 340 nm assay) | Understand what you are measuring |
| 2 | Inspect the pipetting scheme (36 reactions) | Know which conditions you cover |
| 3 | Perform 36 photometric measurements at the bench | Capture initial rates (dA/dt) |
| 4 | Enter your absorbance values via the widget | Get raw data into Python |
| 5 | Compute concentrations & specific activity | Turn raw data into kinetic information |
| 6 | Export a FAIR-ready CSV | Hand off to RDM-01a |

> **No bench access?** A complete reference dataset is provided under `Data/examples/data_initialrate.csv` and is loaded automatically as a fallback if you skip the data-entry section. This lets you continue with RDM-01a and KIN-01a even without performing the experiment yourself.


In [3]:
# Path setup and data directory management
import os
import pandas as pd
from pathlib import Path

# Project layout (this notebook lives in Notebooks/)
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "Data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
EXAMPLES_DIR = DATA_DIR / "examples"
WET01_DIR = DATA_DIR / "WET-01a"          # your own measurements live here

# Make sure all directories exist
for directory in [RAW_DATA_DIR, PROCESSED_DATA_DIR, EXAMPLES_DIR, WET01_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"  Your measurements -> {WET01_DIR}")
print(f"  Example dataset   -> {EXAMPLES_DIR}")
print(f"  Processed output  -> {PROCESSED_DATA_DIR}\n")

# Locate the bundled reference dataset (used as fallback if no own data is entered)
EXAMPLE_DATASET = EXAMPLES_DIR / "data_initialrate.csv"
if EXAMPLE_DATASET.exists():
    print(f"Reference dataset available: {EXAMPLE_DATASET.name}")
    print("  -> If you skip the wet-lab data entry, this file will be used for downstream modules.")
else:
    print("WARNING: reference dataset not found. Please ensure Data/examples/data_initialrate.csv exists.")


Project root: /Users/niklas-maximilianepping/Projekte/TeachOpenBiocatalysis
  Your measurements -> /Users/niklas-maximilianepping/Projekte/TeachOpenBiocatalysis/Data/WET-01a
  Example dataset   -> /Users/niklas-maximilianepping/Projekte/TeachOpenBiocatalysis/Data/examples
  Processed output  -> /Users/niklas-maximilianepping/Projekte/TeachOpenBiocatalysis/Data/processed

Reference dataset available: data_initialrate.csv
  -> If you skip the wet-lab data entry, this file will be used for downstream modules.


In [4]:
# Import required libraries
import numpy as np
import pandas as pd  # for data wrangling
import matplotlib.pyplot as plt  # for visualization
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set up plotting parameters
plt.style.use('default')  # Using default matplotlib style
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

<a id="sec-setup"></a>

## 0. Setup

The two code cells above prepare the Python environment and resolve all data paths. Run them once at the beginning of the session. They are intentionally kept lightweight so that students with limited Python experience are not overwhelmed.

[Back to top](#top)

---

<a id="sec-part1"></a>

# Part 1 — Theoretical Background and Experimental Design

### Learning Objectives

By completing this module, you will be able to:

- Understand the theoretical foundations of enzyme kinetics and spectrophotometric assays
- Design systematic initial-rate experiments for kinetic parameter determination
- Perform accurate data acquisition following STRENDA guidelines
- Calculate substrate / product concentrations from absorbance measurements using the Lambert–Beer law
- Prepare FAIR-compliant datasets for computational analysis and data sharing

### Why study FDH kinetics?

Formate dehydrogenase (FDH) is an industrially relevant enzyme used for **cofactor regeneration** in asymmetric synthesis. Understanding its kinetics is crucial for:

- **Process optimization** — designing efficient biocatalytic cascades
- **Inhibition studies** — managing product inhibition in continuous processes
- **Scale-up** — predicting enzyme performance under production conditions

This hands-on module connects fundamental enzyme kinetics with real-world biocatalysis applications in the pharmaceutical and fine-chemical industries.


<a id="sec-1-1"></a>

## 1.1 FDH-catalyzed NADH cofactor regeneration

In this experiment an enzyme kinetic is determined using the example of formate dehydrogenase (FDH). This enzyme is industrially applied by Degussa AG (Germany) for cofactor regeneration in the production of L-*tert*-leucine on ton-scale by reductive amination catalyzed by leucine dehydrogenase (LeuDH), as shown in Figure 1.


<figure>
  <img src="../Figures/WET-01/WET-01a_ModelReactionHighlighted.png" alt="FDH reaction scheme">
  <figcaption><b>Figure 1:</b> Industrial production of L-tert-Leucine: Enantioselective, reductive amination of trimethylpyruvate catalyzed by leucine dehydrogenase (LeuDH) and regeneration of the oxidized cofactor by formate dehydrogenase (FDH).</figcaption>
</figure>

<a id="sec-1-2"></a>

## 1.2 The FDH reaction and spectrophotometric detection

The FDH-catalyzed reaction proceeds as follows:

$$\text{HCOO}^- + \text{NAD}^+ \xrightarrow{\text{FDH}} \text{CO}_2 + \text{NADH} + \text{H}^+$$

### Thermodynamic considerations

Under the experimental conditions (pH 8.0, open system), the reaction equilibrium strongly favors product formation because:

- CO₂ degassing continuously removes product from the reaction mixture
- The reaction can be treated as pseudo-irreversible for initial-rate measurements
- This simplifies kinetic analysis by avoiding reverse-reaction terms

### Why spectrophotometry at 340 nm?

The key advantage of this assay is the spectroscopic distinction between NAD⁺ and NADH:

- NADH has a strong absorption maximum at **340 nm** ($\varepsilon_{NADH} = 6220~\text{L}\,\text{mol}^{-1}\,\text{cm}^{-1}$)
- NAD⁺ shows negligible absorption at this wavelength (see Figure 2)
- No interference from other assay components (buffer, enzyme, formate, CO₂)

This allows direct, continuous monitoring of reaction progress by measuring the increase in absorbance at 340 nm, which is directly proportional to NADH formation.

### Practical implications

- **Non-destructive:** the same sample can be measured repeatedly
- **Real-time:** reaction progress is monitored continuously
- **Quantitative:** concentration can be calculated precisely using the Lambert–Beer law
- **Simple setup:** standard UV-Vis spectrophotometer with 1 cm cuvettes


<figure>
  <img src="../Figures/WET-01/WET-01a_nad-nadh-absorption.svg">
  <figcaption><b>Figure 2:</b> Absorption spectra of NAD+ and NADH [2].</figcaption>
</figure>

<a id="sec-1-3"></a>

## 1.3 Determination of reactant concentrations

In the FDH assay:

$$\text{HCOO}^- + \text{NAD}^+ \xrightarrow{\text{FDH}} \text{CO}_2 + \text{NADH}$$

Since only NADH absorbs significantly at 340 nm, the change in absorbance $\frac{dA}{dt}$ directly reflects the rate of NADH formation, and thus the rate of the enzymatic reaction.

From the Lambert–Beer law:

$$\frac{dA}{dt} = \varepsilon_{NADH} \cdot d \cdot \frac{dc_{NADH}}{dt}$$

so:

$$\frac{dc_{NADH}}{dt} = \frac{1}{\varepsilon_{NADH} \cdot d} \cdot \frac{dA}{dt}$$

Because the reaction stoichiometry is 1:1:1:1, this also gives the rates of NAD⁺ consumption and formate oxidation, even though they are not directly measurable by absorbance.

<a id="sec-1-4"></a>

## 1.4 Determination of the specific activity of FDH

The specific activity of FDH can be calculated from the formed NADH by the following equation, derived from the Lambert–Beer law:

$$v_{spec} = \frac{1}{\varepsilon_{NADH} \cdot d \cdot c_{FDH}} \cdot \frac{dA}{dt}$$

where:

- $v_{spec}$ is the enzyme specific activity in U·mg⁻¹
- $\frac{dA}{dt}$ is the absorption change per time in min⁻¹
- $\varepsilon_{NADH}$ is the extinction coefficient of NADH in mL·µmol⁻¹·cm⁻¹
- $d$ is the cuvette path length in cm
- $c_{FDH}$ is the enzyme concentration in the assay in mg·mL⁻¹

[Back to top](#top)


<a id="sec-part2"></a>

# Part 2 — Experimental Setup and Protocols

## Experimental Strategy: Initial-Rate Measurements

### What are initial rates?

Initial-rate measurements capture the enzyme's catalytic velocity during the early, linear phase of the reaction (typically < 10 % conversion). At this stage:

- Substrate depletion is negligible: $[S] \approx [S]_0$
- Product accumulation is minimal (and therefore so is product inhibition, unless deliberately added)
- The enzyme is in its active state — no time-dependent deactivation

### Why measure initial rates?

Initial rates provide the cleanest kinetic data because:

1. The **steady-state assumption** holds — the enzyme–substrate complex concentration is constant
2. The **reverse reaction is negligible** — product concentration is too low to drive a backward flux
3. The **absorbance change is linear in time**
4. The **Michaelis–Menten equation** applies directly

### Systematic variation for complete kinetic characterization

To fully characterize FDH kinetics, we systematically vary:

- $[\text{NAD}^+]$: 0–1 mM (co-substrate)
- $[\text{HCOO}^-]$: 0–100 mM (substrate)
- $[\text{NADH}]$: 0–200 µM (product inhibitor)

This **36-measurement design space** allows determination of:

- $K_m^{\text{NAD}^+}$ and $K_m^{\text{HCOO}^-}$ (Michaelis constants)
- $v_{max}$ (maximum velocity)
- $K_i^{\text{NADH}}$ (inhibition constant)
- The inhibition mechanism (competitive vs. non-competitive)

---

## Required Materials and Equipment

Below is the complete list of hardware, consumables and solutions needed for this assay. Following STRENDA guidelines, all conditions are documented to ensure reproducibility.

<a id="sec-2-1"></a>

### 2.1 Hardware

- Photometer ($\lambda = 340$ nm)
- Timer

<a id="sec-2-2"></a>

### 2.2 Consumables

- Pipettes and tips
- UV cuvettes ($d = 1$ cm)

<a id="sec-2-3"></a>

### 2.3 Buffers, Solutions and Experimental Conditions

#### 2.3.1 Experimental conditions

- Temperature: 20–25 °C
- Concentration ranges investigated:
    - $[\text{NAD}^+]$: 0–1 mM
    - $[\text{NADH}]$: 0–200 µM
    - $[\text{HCOO}^-]$: 0–100 mM
    - $[\text{FDH}]$: 40 mg·L⁻¹

#### 2.3.2 Stock solutions

For the activity assay, stock solutions of the reaction components are prepared in potassium-phosphate buffer (50 mM, pH 8.0):

- $[\text{NAD}^+]$: 1.67 mM ($M = 663.4$ g·mol⁻¹, 20 mL)
- $[\text{NADH}]$: 1 mM ($M = 709.4$ g·mol⁻¹, 10 mL)
- $[\text{HCOO}^-]$: 1000 mM ($M = 68$ g·mol⁻¹, 10 mL)
- $[\text{FDH}]$: 800 mg·mL⁻¹ (2.5 mL)

The FDH solution is provided by the supervisor. NADH and NAD⁺ solutions must be prepared freshly and kept on ice. Aliquots of the substances are provided and must be dissolved in potassium-phosphate buffer to the desired concentrations.


<a id="sec-2-4"></a>

### 2.4 Metadata and the STRENDA guidelines

Accurate and complete **metadata** — information describing the conditions and parameters of an assay — are essential for the reproducibility, comparability, and credibility of enzymatic experiments. Even slight deviations in conditions (e.g. temperature, pH, or substrate concentrations) can significantly affect measured rates.

The **STRENDA guidelines** (Standards for Reporting Enzymology Data) provide a community-agreed framework ensuring that kinetic data are reported in a transparent, reproducible and reusable way. Adhering to these standards aligns with the **FAIR principles** and supports the integration of data into community databases such as **BRENDA**, **STRENDA DB** or **SABIO-RK**.

In this module we apply these principles to your FDH assay by documenting all STRENDA-compliant metadata necessary for later modeling. A more detailed discussion of research data management follows in module **RDM-01a**.


In [5]:
# Meta-data (from the experimental protocol) according to STRENDA-DB
meta_data = {
    # stock-solutions used
    "NAD_stock_mM":                 1.081993569,    # need to be modified / changed accordingly
    "HCOO_stock_mM":                1000.0,
    "NADH_stock_mM":                0.869453376,    # need to be modified / changed accordingly
    "FDH_stock_mg_per_L":           800.0,
    
    # assay-specific meta-data
    'buffer':                       '50 mM Potassium-phosphate buffer',
    'pH':                           8.0,
    'assay_volume_mL':              1.0,
    'temperature_C':                22,             # room temperature 20-25 °C   
    "cuvette_path_cm":              1.0,
    'wavelength_nm':                340,            # NADH absorption maximum / detection wavelength
    "epsilon_NADH_L_per_mol_cm":    6220.0,
    
    # enzyme / reaction specific meta-data
    'enzyme_uniprot': 'O13437',                     # UniProt ID
    'enzyme_name': 'Formate dehydrogenase (FDH1) from _Candida boidinii_',  # arbitrary name given to enzyme
    'enzyme_concentration_mg_per_L': 40.0,          # from course use case
    'enzyme_mass_da': 40370,                        # from UniProt
    'enzyme_concentration_uM': None,                # convert if molecular weight known
    'enzyme_catalyzed_reaction_rhea': 15985,        # Rhea reaction ID
}

# Derived values
meta_data["epsilon_NADH_mL_per_umol_cm"] = meta_data["epsilon_NADH_L_per_mol_cm"] / 1000
meta_data

{'NAD_stock_mM': 1.081993569,
 'HCOO_stock_mM': 1000.0,
 'NADH_stock_mM': 0.869453376,
 'FDH_stock_mg_per_L': 800.0,
 'buffer': '50 mM Potassium-phosphate buffer',
 'pH': 8.0,
 'assay_volume_mL': 1.0,
 'temperature_C': 22,
 'cuvette_path_cm': 1.0,
 'wavelength_nm': 340,
 'epsilon_NADH_L_per_mol_cm': 6220.0,
 'enzyme_uniprot': 'O13437',
 'enzyme_name': 'Formate dehydrogenase (FDH1) from _Candida boidinii_',
 'enzyme_concentration_mg_per_L': 40.0,
 'enzyme_mass_da': 40370,
 'enzyme_concentration_uM': None,
 'enzyme_catalyzed_reaction_rhea': 15985,
 'epsilon_NADH_mL_per_umol_cm': 6.22}

In [6]:
# Load the pipetting scheme that defines the 36 experimental conditions
pipetting_path = WET01_DIR / "pipetting_scheme.csv"
pipetting_df = pd.read_csv(pipetting_path)

print(f"Loaded pipetting scheme from: {pipetting_path.relative_to(PROJECT_ROOT)}")
print(f"Number of conditions: {len(pipetting_df)}")
display(pipetting_df)


Loaded pipetting scheme from: Data/WET-01a/pipetting_scheme.csv
Number of conditions: 36


,NAD_µL,HCOO_µL,NADH_µL,FDH_µL,Buffer_µL
0,600,0,0,50,350
1,600,5,0,50,345
2,600,10,0,50,340
3,600,20,0,50,330
4,600,100,0,50,250
5,600,200,0,50,150
6,0,200,0,50,750
7,10,200,0,50,740
8,20,200,0,50,730
9,75,200,0,50,675


<a id="sec-2-5"></a>

### 2.5 Pipetting scheme and initial concentrations

Concentrations after pipetting are obtained by simple dilution:

$$c_\text{final} = \frac{V_\text{stock}}{V_\text{total}} \cdot c_\text{stock}$$

where all volumes are in µL and concentrations in mM.

[Back to top](#top)


In [7]:
# Constants
V_total = 1000  # µL
conv = 1 / V_total

# Calculate initial concentrations (in mM) for all substrates
initial_conc = pd.DataFrame({
    "FDH_mg_per_mL": (pipetting_df["FDH_µL"] * meta_data["FDH_stock_mg_per_L"] / 1000) * conv,
    "NAD_mM": pipetting_df["NAD_µL"] * meta_data["NAD_stock_mM"] * conv,
    "HCOO_mM": pipetting_df["HCOO_µL"] * meta_data["HCOO_stock_mM"] * conv,
    "NADH_mM": pipetting_df["NADH_µL"] * meta_data["NADH_stock_mM"] * conv,
})

initial_conc.round(5).head()

,FDH_mg_per_mL,NAD_mM,HCOO_mM,NADH_mM
0,0.04,0.6492,0.0,0.0
1,0.04,0.6492,5.0,0.0
2,0.04,0.6492,10.0,0.0
3,0.04,0.6492,20.0,0.0
4,0.04,0.6492,100.0,0.0


---

<a id="sec-part3"></a>

# Part 3 — Data Acquisition

## Overview: From Pipetting to Absorbance Measurement

In this section you will execute the experimental protocol to collect initial-rate data for 36 different substrate / inhibitor combinations. Each measurement captures the enzyme's velocity under precisely controlled conditions.

### The measurement workflow

For each of the 36 assays you will:

1. Prepare the reaction mixture according to the pipetting scheme (substrates + buffer)
2. Blank the spectrophotometer using a reference cuvette (all components except enzyme)
3. Measure the initial absorbance $A_0$ before adding enzyme
4. Start the reaction by adding the FDH enzyme solution
5. Measure the absorbance after 60 s ($A_{60}$)
6. Record the data in the entry widget below

### Why two absorbance measurements?

The difference between $A_{60}$ and $A_0$ represents the NADH formed solely by enzymatic activity:

$$\Delta A = A_{60} - A_0$$

This accounts for:

- Any pre-existing NADH in the substrate stocks
- Background absorbance from buffer components
- Baseline drift in the spectrophotometer

Without measuring $A_0$ we could not distinguish between pre-existing and newly formed NADH — critical for an accurate rate determination.


<a id="sec-3-1"></a>

## 3.1 The Measurement Protocol — Step by Step

For **each of the 36 cuvettes** defined in the pipetting scheme:

1. **Blank** the spectrophotometer at 340 nm with buffer only (auto-zero).
2. **Prepare the cuvette** with all components *except* the enzyme (NAD⁺ + HCOO⁻ + NADH + buffer).
3. **Record A₀** — the initial absorbance (captures any NADH present before the reaction starts).
4. **Add FDH** (50 µL of stock), mix quickly by inversion, and **start the timer**.
5. **Record A₆₀** — the absorbance at exactly 60 s after enzyme addition.
6. **Enter the value into the widget below** (cell 3.2).

The enzymatic NADH formation is then:

$$\Delta A = A_{60} - A_{0} \quad\Rightarrow\quad \frac{dA}{dt} \approx \frac{\Delta A}{60~\text{s}} = \Delta A \cdot 1~\text{min}^{-1}$$

**Common pitfalls to avoid**

- Make sure $\Delta A < 0.1$ — otherwise you are outside the linear range of the photometer.
- For very fast samples (high [NAD⁺] + high [HCOO⁻]), shorten the measurement window or dilute the enzyme.
- Always record $A_0$ separately, even if you expect it to be ≈ 0 — without it you cannot subtract pre-existing NADH or buffer drift.

---

<a id="sec-3-2"></a>

## 3.2 Entering Your Absorbance Values

The widget below lets you **add, edit and review** your measured absorbance values, one sample at a time.

**How to use it:**

| Action | What to do |
|--------|------------|
| Add a new value | Select the *Sample ID* (1–36), type the *A₀* and *A₆₀* values, optionally a *Comment*, then click **Save / Update**. |
| Correct an entry | Re-select the same *Sample ID*, type the new value, click **Save / Update** — the row is overwritten. |
| Inspect progress | Click **Show table** to see all 36 rows; missing values show as `NaN`. |
| Reset everything | Click **Reset** (asks for confirmation by requiring a second click within 5 s). |

All entries are **persisted immediately** to `Data/WET-01a/absorbance_data.csv`, so you don't lose data if the kernel restarts.

> **No bench access?** Just skip cell 3.2 — the next cell (3.3) automatically falls back to the bundled reference dataset `Data/examples/data_initialrate.csv`, which contains a complete set of 36 measurements. RDM-01a and KIN-01a will work either way.



In [8]:
# -----------------------------------------------------------------------------
# Interactive data entry widget for absorbance values
# -----------------------------------------------------------------------------
# Persistent file – your raw measurements live here:
import ipywidgets as widgets
from IPython.display import display, clear_output
import time
import numpy as np

ABS_DATA_PATH = WET01_DIR / "absorbance_data.csv"
N_SAMPLES = 36

# 1. Load or initialize the absorbance table -----------------------------------
def _empty_table(n=N_SAMPLES):
    return pd.DataFrame({"ID": range(1, n + 1),
                         "Abs_0s":  [np.nan] * n,
                         "Abs_60s": [np.nan] * n,
                         "Comment": [""] * n})

if ABS_DATA_PATH.exists():
    abs_df = pd.read_csv(ABS_DATA_PATH)
    # Make sure schema is up-to-date (older files might miss Abs_0s)
    for col in ["Abs_0s", "Abs_60s", "Comment"]:
        if col not in abs_df.columns:
            abs_df[col] = np.nan if col != "Comment" else ""
    if len(abs_df) < N_SAMPLES:
        missing = _empty_table(N_SAMPLES).iloc[len(abs_df):]
        abs_df = pd.concat([abs_df, missing], ignore_index=True)
    print(f"Loaded existing data: {ABS_DATA_PATH.name} "
          f"({abs_df['Abs_60s'].notna().sum()}/{N_SAMPLES} values filled)")
else:
    abs_df = _empty_table()
    abs_df.to_csv(ABS_DATA_PATH, index=False)
    print(f"Created empty data file: {ABS_DATA_PATH.name}")

# 2. Build the widget ----------------------------------------------------------
id_sel = widgets.BoundedIntText(value=1, min=1, max=N_SAMPLES,
                                description="Sample ID:", style={"description_width": "100px"})
a0_in  = widgets.FloatText(description="A0 (t=0):", step=0.001,
                           style={"description_width": "100px"})
a60_in = widgets.FloatText(description="A60 (t=60s):", step=0.001,
                           style={"description_width": "100px"})
cmt_in = widgets.Text(description="Comment:",
                      style={"description_width": "100px"},
                      layout=widgets.Layout(width="450px"))

save_btn  = widgets.Button(description="Save / Update", button_style="success")
show_btn  = widgets.Button(description="Show table",   button_style="info")
reset_btn = widgets.Button(description="Reset",        button_style="danger")
out       = widgets.Output()

_reset_armed = {"ts": 0.0}

def _load_row_into_form(*_):
    """Pre-fill the form with the currently stored values for the selected ID."""
    row = abs_df.loc[abs_df["ID"] == id_sel.value].iloc[0]
    a0_in.value  = 0.0 if pd.isna(row["Abs_0s"])  else float(row["Abs_0s"])
    a60_in.value = 0.0 if pd.isna(row["Abs_60s"]) else float(row["Abs_60s"])
    cmt_in.value = "" if pd.isna(row["Comment"]) else str(row["Comment"])

id_sel.observe(_load_row_into_form, names="value")
_load_row_into_form()  # initial fill

def _on_save(_):
    idx = abs_df.index[abs_df["ID"] == id_sel.value][0]
    abs_df.at[idx, "Abs_0s"]  = a0_in.value
    abs_df.at[idx, "Abs_60s"] = a60_in.value
    abs_df.at[idx, "Comment"] = cmt_in.value
    abs_df.to_csv(ABS_DATA_PATH, index=False)
    filled = abs_df["Abs_60s"].notna().sum()
    with out:
        clear_output()
        print(f"Saved sample {id_sel.value}: "
              f"A0={a0_in.value:.4f}, A60={a60_in.value:.4f}  "
              f"({filled}/{N_SAMPLES} samples entered)")
    # auto-advance to next empty sample for convenience
    next_empty = abs_df.loc[abs_df["Abs_60s"].isna(), "ID"]
    if len(next_empty) > 0:
        id_sel.value = int(next_empty.iloc[0])

def _on_show(_):
    with out:
        clear_output()
        display(abs_df)

def _on_reset(_):
    """Two-click safety: arm on first click, execute on second click within 5 s."""
    global abs_df
    now = time.time()
    if now - _reset_armed["ts"] > 5:
        _reset_armed["ts"] = now
        with out:
            clear_output()
            print("Click 'Reset' again within 5 s to confirm wiping ALL entries.")
        return
    abs_df = _empty_table()
    abs_df.to_csv(ABS_DATA_PATH, index=False)
    _reset_armed["ts"] = 0.0
    _load_row_into_form()
    with out:
        clear_output()
        print("All entries cleared.")

save_btn.on_click(_on_save)
show_btn.on_click(_on_show)
reset_btn.on_click(_on_reset)

ui = widgets.VBox([
    widgets.HTML("<b>Absorbance data entry</b> &mdash; values are saved to "
                 f"<code>{ABS_DATA_PATH.relative_to(PROJECT_ROOT)}</code> on every save."),
    id_sel, a0_in, a60_in, cmt_in,
    widgets.HBox([save_btn, show_btn, reset_btn]),
    out,
])
display(ui)



Loaded existing data: absorbance_data.csv (0/36 values filled)


<a id="sec-3-3"></a>

## 3.3 Load measurements (with automatic fallback to reference data)

The cell below loads the absorbance values you just entered. If your file is **empty or incomplete**, it automatically falls back to the **bundled reference dataset** `Data/examples/data_initialrate.csv` so the rest of the notebook (and RDM-01a, KIN-01a) still works end-to-end. A clear message tells you which source is being used.



In [9]:
# -----------------------------------------------------------------------------
# Load the measurements – use student-entered data if complete, else fall back
# -----------------------------------------------------------------------------
USE_EXAMPLE_DATA = False
abs_df_user = pd.read_csv(ABS_DATA_PATH) if ABS_DATA_PATH.exists() else _empty_table()

n_filled = abs_df_user["Abs_60s"].notna().sum() if "Abs_60s" in abs_df_user.columns else 0

if n_filled == N_SAMPLES:
    print(f"Using YOUR measurements from {ABS_DATA_PATH.name} ({n_filled}/{N_SAMPLES} complete).")
    abs_df = abs_df_user.copy()
    # Compute dA/dt in min^-1 from delta_A over 60 s (= 1 min) — student data
    a0  = abs_df["Abs_0s"].fillna(0.0)
    a60 = abs_df["Abs_60s"]
    abs_df["dA_per_min"] = (a60 - a0) / 1.0   # 60 s == 1 min
else:
    USE_EXAMPLE_DATA = True
    print(f"Only {n_filled}/{N_SAMPLES} measurements entered "
          f"-> falling back to reference dataset.")
    print(f"   Source: {EXAMPLE_DATASET.relative_to(PROJECT_ROOT)}")
    print("   (RDM-01a and KIN-01a will use this dataset if you don't add your own.)\n")

    # Reference dataset uses European format: ';' separator, ',' decimal, 2-row header
    ref = pd.read_csv(EXAMPLE_DATASET, sep=';', decimal=',', header=[0, 1])
    # Flatten the multi-header -> keep only the variable names
    ref.columns = [c[0] for c in ref.columns]
    # Re-cast the columns we need
    abs_df = pd.DataFrame({
        "ID": range(1, len(ref) + 1),
        "Abs_0s":  0.0,                                  # reference assumes A0 = 0
        "Abs_60s": ref["initial_rate"].astype(float),    # delta_A over 1 min equals dA/dt
        "dA_per_min": ref["initial_rate"].astype(float),
        "Comment": ["reference dataset"] * len(ref),
    })

print("\nPreview:")
display(abs_df.head())



Only 0/36 measurements entered -> falling back to reference dataset.
   Source: Data/examples/data_initialrate.csv
   (RDM-01a and KIN-01a will use this dataset if you don't add your own.)


Preview:


,ID,Abs_0s,Abs_60s,dA_per_min,Comment
0,1,0.0,0.0005,0.0005,reference dataset
1,2,0.0,0.0067,0.0067,reference dataset
2,3,0.0,0.0142,0.0142,reference dataset
3,4,0.0,0.0368,0.0368,reference dataset
4,5,0.0,0.0546,0.0546,reference dataset


In [10]:
# -----------------------------------------------------------------------------
# Assemble the working DataFrame for Part 4
#
# Source of truth:
#   - abs_df       : raw photometer readout (your own data OR the reference set)
#                    columns: ID, Abs_0s, Abs_60s, dA_per_min, Comment
#   - initial_conc : initial concentrations derived from the pipetting scheme
#                    columns: FDH_mg_per_mL, NAD_mM, HCOO_mM, NADH_mM
#
# We merge them into a single tidy `data_df` that downstream cells operate on.
# This way Part 4 always reflects whatever was loaded in cell 3.3 – student
# data when present, reference dataset otherwise.
# -----------------------------------------------------------------------------

if len(abs_df) != len(initial_conc):
    raise ValueError(
        f"Row count mismatch: abs_df has {len(abs_df)} rows but "
        f"initial_conc has {len(initial_conc)} rows. Re-run cells 2.5 and 3.3."
    )

data_df = pd.DataFrame({
    "ID":           abs_df["ID"].values,
    "FDH_0":        initial_conc["FDH_mg_per_mL"].values,   # mg/mL
    "NAD_0":        initial_conc["NAD_mM"].values,          # mM
    "HCOO_0":       initial_conc["HCOO_mM"].values,         # mM
    "NADH_0":       initial_conc["NADH_mM"].values,         # mM
    "CO2_0":        0.0,                                    # assumed degassed
    "initial_rate": abs_df["dA_per_min"].values,            # min^-1
    "Abs_0s":       abs_df["Abs_0s"].values,
    "Abs_60s":      abs_df["Abs_60s"].values,
})

# Keep the units around for the FAIR export later
column_units = {
    "ID":           "-",
    "FDH_0":        "mg/mL",
    "NAD_0":        "mM",
    "HCOO_0":       "mM",
    "NADH_0":       "mM",
    "CO2_0":        "mM",
    "initial_rate": "1/min",
    "Abs_0s":       "AU",
    "Abs_60s":      "AU",
}

src = "reference dataset" if USE_EXAMPLE_DATA else "your own measurements"
print(f"Working dataset assembled from {src} ({len(data_df)} rows).")
print("\nColumn overview:")
for name, unit in column_units.items():
    print(f"  {name:<14} [{unit}]")

print("\nPreview (first 5 rows):")
display(data_df.head())


Working dataset assembled from reference dataset (36 rows).

Column overview:
  ID             [-]
  FDH_0          [mg/mL]
  NAD_0          [mM]
  HCOO_0         [mM]
  NADH_0         [mM]
  CO2_0          [mM]
  initial_rate   [1/min]
  Abs_0s         [AU]
  Abs_60s        [AU]

Preview (first 5 rows):


,ID,FDH_0,NAD_0,HCOO_0,NADH_0,CO2_0,initial_rate,Abs_0s,Abs_60s
0,1,0.04,0.649196,0.0,0.0,0.0,0.0005,0.0,0.0005
1,2,0.04,0.649196,5.0,0.0,0.0,0.0067,0.0,0.0067
2,3,0.04,0.649196,10.0,0.0,0.0,0.0142,0.0,0.0142
3,4,0.04,0.649196,20.0,0.0,0.0,0.0368,0.0,0.0368
4,5,0.04,0.649196,100.0,0.0,0.0,0.0546,0.0,0.0546


---

<a id="sec-part4"></a>

# Part 4 — Data Processing and Analysis

## From Absorbance to Kinetic Parameters

Now that you have collected raw absorbance data for all 36 assays (or are using the bundled reference dataset), it is time to **transform measurements into meaningful kinetic information**. This section shows how to:

1. Apply the **Lambert–Beer law** to calculate NADH concentrations
2. Use **reaction stoichiometry** to determine all species concentrations
3. Calculate the **specific enzyme activity** (U/mg)
4. Prepare a **FAIR-compliant dataset** for downstream analysis

These calculations bridge the gap between **raw experimental data** and the **quantitative parameters** needed for kinetic modeling in module KIN-01a.

---

<a id="sec-4-1"></a>

## 4.1 Theoretical Background for the Calculations

### 1. Lambert–Beer law for NADH concentration

The Lambert–Beer law relates absorbance to concentration:

$$A = \varepsilon \cdot c \cdot d$$

where:

- $A$ = absorbance (dimensionless)
- $\varepsilon$ = molar extinction coefficient ($6220~\text{L}\,\text{mol}^{-1}\,\text{cm}^{-1}$ for NADH at 340 nm)
- $c$ = concentration ($\text{mol}\,\text{L}^{-1}$)
- $d$ = path length ($1~\text{cm}$ for standard cuvettes)

Rearranging for concentration:

$$c_\text{NADH} = \frac{A}{\varepsilon \cdot d}$$

### 2. Converting the initial rate into a concentration change

The `initial_rate` column contains $\frac{dA}{dt}$ in units of $\text{min}^{-1}$. For a 60-second (1 min) measurement:

$$\Delta A = \frac{dA}{dt} \cdot 1~\text{min}$$

This absorbance change corresponds to the NADH formed during the reaction.

### 3. Reaction stoichiometry

The FDH-catalysed reaction has 1:1:1:1 stoichiometry:

$$\text{HCOO}^- + \text{NAD}^+ \xrightarrow{\text{FDH}} \text{CO}_2 + \text{NADH}$$

Therefore:

- $[\text{NADH}]_{60} = [\text{NADH}]_0 + \Delta[\text{NADH}]$
- $[\text{NAD}^+]_{60} = [\text{NAD}^+]_0 - \Delta[\text{NADH}]$
- $[\text{HCOO}^-]_{60} = [\text{HCOO}^-]_0 - \Delta[\text{NADH}]$
- $[\text{CO}_2]_{60} = [\text{CO}_2]_0 + \Delta[\text{NADH}]$

### 4. Specific enzyme activity (U/mg)

Enzyme activity is defined as the amount of enzyme that converts 1 µmol of substrate per minute under specified conditions:

$$\text{Specific activity} = \frac{v_\text{NADH}}{m_\text{enzyme}}$$

where

- $v_\text{NADH}$ = absolute rate of NADH formation ($\mu\text{mol}\,\text{min}^{-1}$)
- $m_\text{enzyme}$ = total enzyme mass in the assay ($\text{mg}$)

**Step-by-step unit conversion**

1. Concentration rate from Lambert–Beer:

$$\frac{dc}{dt} = \frac{1}{\varepsilon \cdot d} \cdot \frac{dA}{dt}\quad[\text{mol}\,\text{L}^{-1}\,\text{min}^{-1}]$$

2. Convert to an absolute rate by multiplying by the assay volume and converting moles to µmol:

$$v_\text{NADH} = \frac{dc}{dt} \cdot V_\text{assay}\,[\text{L}] \cdot 10^6\quad[\mu\text{mol}\,\text{min}^{-1}]$$

3. Specific activity:

$$\text{Specific activity} = \frac{v_\text{NADH}~[\mu\text{mol}\,\text{min}^{-1}]}{c_\text{enzyme}~[\text{mg}\,\text{mL}^{-1}] \cdot V_\text{assay}~[\text{mL}]}\quad[\text{U}\,\text{mg}^{-1}]$$

[Back to top](#top)



In [11]:
# Extract experimental parameters from the metadata dictionary
epsilon_NADH = meta_data["epsilon_NADH_L_per_mol_cm"]   # L*mol^-1*cm^-1
path_length  = meta_data["cuvette_path_cm"]             # cm
assay_volume = meta_data["assay_volume_mL"]             # mL

print("Experimental parameters:")
print(f"   epsilon(NADH) = {epsilon_NADH} L/mol/cm")
print(f"   path length   = {path_length} cm")
print(f"   assay volume  = {assay_volume} mL\n")

# Step 1: Lambert-Beer  --------------------------------------------------------
# Absorbance change in 60 s == 1 min: delta_A = (dA/dt) * 1 min
data_df["dA_60s"] = data_df["initial_rate"] * 1.0

# c = A / (epsilon * d); factor 1000 converts mol/L to mmol/L (mM)
data_df["dNADH_mM"] = (data_df["dA_60s"] / (epsilon_NADH * path_length)) * 1000
data_df["NADH_60"] = data_df["NADH_0"] + data_df["dNADH_mM"]

print("Step 1: NADH concentration computed via Lambert-Beer.")

# Step 2: Stoichiometry  -------------------------------------------------------
# HCOO- + NAD+ -> CO2 + NADH  (1:1:1:1)
data_df["NAD_60"]  = data_df["NAD_0"]  - data_df["dNADH_mM"]
data_df["HCOO_60"] = data_df["HCOO_0"] - data_df["dNADH_mM"]
data_df["CO2_60"]  = data_df["CO2_0"]  + data_df["dNADH_mM"]

print("Step 2: substrate/product concentrations from stoichiometry.")

# Step 3: Specific activity (U/mg)  --------------------------------------------
# dc/dt [mol/L/min]
conc_rate_mol_per_L_min = data_df["initial_rate"] / (epsilon_NADH * path_length)

# v_NADH [µmol/min] = dc/dt * V[L] * 1e6
assay_volume_L = assay_volume / 1000.0
rate_NADH_umol_per_min = conc_rate_mol_per_L_min * assay_volume_L * 1e6

# enzyme mass [mg] = c_enzyme [mg/mL] * V [mL]
enzyme_mass_mg = data_df["FDH_0"] * assay_volume

# U/mg = (µmol/min) / mg
data_df["specific_activity_U_per_mg"] = rate_NADH_umol_per_min / enzyme_mass_mg

print("Step 3: specific enzyme activity computed.\n")

# Summary ----------------------------------------------------------------------
print("=" * 70)
print("CALCULATION SUMMARY")
print("=" * 70)
print(f"Mean delta_NADH:           {data_df['dNADH_mM'].mean():.4f} +/- {data_df['dNADH_mM'].std():.4f} mM")
print(f"Mean specific activity:    {data_df['specific_activity_U_per_mg'].mean():.4f} +/- {data_df['specific_activity_U_per_mg'].std():.4f} U/mg")
print(f"Max specific activity:     {data_df['specific_activity_U_per_mg'].max():.4f} U/mg")
print("=" * 70)

calculated_cols = ["dA_60s", "dNADH_mM",
                   "NAD_60", "HCOO_60", "NADH_60", "CO2_60",
                   "specific_activity_U_per_mg"]
print("\nPreview of the calculated data (first 5 rows):")
display(data_df[["NAD_0", "HCOO_0", "NADH_0", "initial_rate"] + calculated_cols].head())


Experimental parameters:
   epsilon(NADH) = 6220.0 L/mol/cm
   path length   = 1.0 cm
   assay volume  = 1.0 mL

Step 1: NADH concentration computed via Lambert-Beer.
Step 2: substrate/product concentrations from stoichiometry.
Step 3: specific enzyme activity computed.

CALCULATION SUMMARY
Mean delta_NADH:           0.0043 +/- 0.0034 mM
Mean specific activity:    0.1071 +/- 0.0846 U/mg
Max specific activity:     0.2576 U/mg

Preview of the calculated data (first 5 rows):


,NAD_0,HCOO_0,NADH_0,initial_rate,dA_60s,dNADH_mM,NAD_60,HCOO_60,NADH_60,CO2_60,specific_activity_U_per_mg
0,0.649196,0.0,0.0,0.0005,0.0005,0.000080,0.649116,-0.000080,0.000080,0.000080,0.002010
1,0.649196,5.0,0.0,0.0067,0.0067,0.001077,0.648119,4.998923,0.001077,0.001077,0.026929
2,0.649196,10.0,0.0,0.0142,0.0142,0.002283,0.646913,9.997717,0.002283,0.002283,0.057074
3,0.649196,20.0,0.0,0.0368,0.0368,0.005916,0.643280,19.994084,0.005916,0.005916,0.147910
4,0.649196,100.0,0.0,0.0546,0.0546,0.008778,0.640418,99.991222,0.008778,0.008778,0.219453


In [12]:
# Complete dataset with all columns -------------------------------------------
print("Complete dataset with calculated values")
print("=" * 70)
print("Columns:")
print("  Initial conditions : NAD_0, HCOO_0, NADH_0, CO2_0, FDH_0")
print("  Measured           : initial_rate (dA/min)")
print("  Calculated         : NAD_60, HCOO_60, NADH_60, CO2_60")
print("  Enzyme activity    : specific_activity_U_per_mg")
print("=" * 70)

output_cols = [
    "ID", "FDH_0",
    "NAD_0", "HCOO_0", "NADH_0", "CO2_0",
    "initial_rate", "dA_60s", "dNADH_mM",
    "NAD_60", "HCOO_60", "NADH_60", "CO2_60",
    "specific_activity_U_per_mg",
]
display(data_df[output_cols])

# Export to CSV for downstream modules (RDM-01a, KIN-01a) ---------------------
export_path = PROCESSED_DATA_DIR / "FDH_kinetic_data_calculated.csv"
data_df.to_csv(export_path, index=False)
print(f"\nData exported to: {export_path.relative_to(PROJECT_ROOT)}")


Complete dataset with calculated values
Columns:
  Initial conditions : NAD_0, HCOO_0, NADH_0, CO2_0, FDH_0
  Measured           : initial_rate (dA/min)
  Calculated         : NAD_60, HCOO_60, NADH_60, CO2_60
  Enzyme activity    : specific_activity_U_per_mg


,ID,FDH_0,NAD_0,HCOO_0,NADH_0,CO2_0,initial_rate,dA_60s,dNADH_mM,NAD_60,HCOO_60,NADH_60,CO2_60,specific_activity_U_per_mg
0,1,0.04,0.649196,0.0,0.000000,0.0,0.0005,0.0005,0.000080,0.649116,-0.000080,0.000080,0.000080,0.002010
1,2,0.04,0.649196,5.0,0.000000,0.0,0.0067,0.0067,0.001077,0.648119,4.998923,0.001077,0.001077,0.026929
2,3,0.04,0.649196,10.0,0.000000,0.0,0.0142,0.0142,0.002283,0.646913,9.997717,0.002283,0.002283,0.057074
3,4,0.04,0.649196,20.0,0.000000,0.0,0.0368,0.0368,0.005916,0.643280,19.994084,0.005916,0.005916,0.147910
4,5,0.04,0.649196,100.0,0.000000,0.0,0.0546,0.0546,0.008778,0.640418,99.991222,0.008778,0.008778,0.219453
5,6,0.04,0.649196,200.0,0.000000,0.0,0.0641,0.0641,0.010305,0.638891,199.989695,0.010305,0.010305,0.257637
6,7,0.04,0.000000,200.0,0.000000,0.0,0.0001,0.0001,0.000016,-0.000016,199.999984,0.000016,0.000016,0.000402
7,8,0.04,0.010820,200.0,0.000000,0.0,0.0210,0.0210,0.003376,0.007444,199.996624,0.003376,0.003376,0.084405
8,9,0.04,0.021640,200.0,0.000000,0.0,0.0270,0.0270,0.004341,0.017299,199.995659,0.004341,0.004341,0.108521
9,10,0.04,0.081150,200.0,0.000000,0.0,0.0482,0.0482,0.007749,0.073400,199.992251,0.007749,0.007749,0.193730



Data exported to: Data/processed/FDH_kinetic_data_calculated.csv


---

<a id="sec-part5"></a>

# Part 5 — FAIR Data Export and Module Integration

## Creating a Reusable, Standards-Compliant Dataset

You have now:

- Acquired 36 initial-rate measurements (or used the bundled reference dataset)
- Applied the Lambert–Beer law to obtain concentrations
- Computed the specific enzyme activity (U/mg)
- Captured all conditions in a STRENDA-style metadata dictionary

The exported CSV is the **handover point** for the rest of the series.

---

## 5.1 What gets exported

| File | Contents | Used by |
|------|----------|---------|
| `Data/WET-01a/absorbance_data.csv` | Your raw $A_0$ / $A_{60}$ values + comments | this notebook |
| `Data/processed/FDH_kinetic_data_calculated.csv` | Concentrations at $t = 0$ / $t = 60~\text{s}$ and specific activity | **RDM-01a** (EnzymeML), **KIN-01a** (modeling) |
| `Data/examples/data_initialrate.csv` | Reference dataset (bundled fallback) | downstream modules if no own data is provided |

If you did not enter your own data, the processed CSV is generated from the reference dataset and is still fully valid for downstream modules — it is just labelled as such in the metadata.

---

## 5.2 Quality check before handing off

Before you move on, verify:

- All 36 measurements are present (no `NaN` in `dA_per_min`)
- Specific activities fall in a plausible range (about 0.05 – 0.15 U/mg for this enzyme batch)
- Initial substrate concentrations match the pipetting scheme
- The metadata dictionary still reflects the actual day-of-experiment conditions (T, pH, stock concentrations, operator)

---

## 5.3 A README for your dataset

A FAIR dataset is never "just the numbers". Together with the CSV you should ship a short **README** that explains:

1. **What** the data represent (FDH initial-rate assay, NADH at 340 nm)
2. **Who** generated them and **when**
3. **How** they were obtained (this notebook, instrument, batch numbers)
4. **Which** STRENDA fields are documented (List 1a + 1b: enzyme, conditions, substrates, kinetics)
5. **How** to cite or reuse the data (license, ORCID, DOI if archived)

RDM-01a picks this up and converts it into a fully machine-readable EnzymeML document.

[Back to top](#top)



<a id="sec-outlook"></a>

# Conclusion & Outlook

## Key learning outcomes

In this module you have:

1. **Understood enzyme kinetics theory**
   - The role of FDH in industrial biocatalysis (cofactor regeneration)
   - Spectrophotometric detection principles (Lambert–Beer law)
   - Initial-rate measurements and their advantages

2. **Performed systematic experiments**
   - Designed a 36-measurement experiment covering substrate and inhibitor space
   - Executed STRENDA-compliant protocols with complete metadata documentation
   - Collected absorbance data using proper blanking and baseline correction

3. **Pre-processed the data for a downstream FAIR-compliant dataset**
   - Applied the Lambert–Beer law to calculate NADH concentrations from raw data
   - Used reaction stoichiometry to determine all species concentrations
   - Calculated specific enzyme activity with proper unit conversions
   - Exported a structured dataset for downstream analysis
   - Documented all experimental conditions following the STRENDA biocatalysis guidelines



## Next steps in the Teach Open Biocatalysis workflow

You are now ready to proceed to **RDM-01a: EnzymeML Data Management**, which transforms your CSV data into a FAIR-compliant EnzymeML document that:

- integrates with community databases (STRENDA DB, BRENDA)
- uses standardised ontologies for machine-readability
- enables automated data exchange with modeling tools



## Reflective Questions

Before moving on, consider:
1. How does systematic variation of substrates/inhibitors enable comprehensive kinetic characterization?
2. Why is STRENDA-compliant metadata documentation crucial for reproducible science?
3. What are the advantages of initial rate measurements over full time-course experiments?

<a id="sec-references"></a>

# References & Resources

1. **FDH-catalyzed reaction (Rhea)**: https://www.rhea-db.org/rhea/15985  
   Standardised reaction database with stoichiometry and EC numbers.
2. **FDH1 from *Candida boidinii* (UniProt)**: https://www.uniprot.org/uniprotkb/O13437/entry  
   Complete enzyme annotation including sequence, structure, and catalytic properties.
3. **NADH/NAD⁺ absorption spectra**: https://commons.wikimedia.org/wiki/File:NADNADH.svg  
   Visualization of spectroscopic differences enabling 340 nm detection.
4. **STRENDA Guidelines**: https://www.beilstein-institut.de/en/projects/strenda/guidelines/  
   Standards for Reporting Enzymology Data (minimum information requirements).
5. **STRENDA DB**: https://www.beilstein-strenda-db.org/  
   Public repository for enzyme functional data.
6. **EnzymeML**: https://enzymeml.org/  
   FAIR data standard for enzyme kinetics (used in modules RDM-01a and KIN-01a).
7. **FAIR Principles**: https://www.go-fair.org/fair-principles/  
   Findable, Accessible, Interoperable, Reusable data guidelines.
8. **Teach Open Biocatalysis**: https://github.com/TeachOpenBiocatalysis  
   Open-source educational materials for biocatalysis research and teaching.

[Back to top](#top)



<a id="sec-ack"></a>

# Acknowledgments

This module was developed as part of the **Teach Open Biocatalysis** initiative — a collaborative effort by **Max Häußler** (University of Stuttgart), **Niklas-Maximilian Epping** (Hamburg University of Technology) and **Amalie Vang Høst** (University of Bielefeld) — to provide accessible, standards-compliant educational materials for enzyme kinetics, research data management and kinetic modeling.

The FDH initial-rate assay presented here has been part of the practical biocatalysis course at the **Institute of Technical Biocatalysis (Prof. Andreas Liese)** at Hamburg University of Technology for many years. We gratefully acknowledge the original designers of the experiment as well as the many staff members, doctoral researchers and student assistants who have continuously refined the protocol, the pipetting scheme and the supporting teaching material over generations of students. Their cumulative effort is the foundation on which this notebook is built.

The downstream modules of this series (**RDM-01a** and **KIN-01a**) build on the research-data-management and modeling tools developed by the **group of Prof. Jürgen Pleiss** at the University of Stuttgart — in particular **Jan Range** and **Torsten Giess**, whose work on EnzymeML and related tooling makes reproducible, FAIR-compliant enzyme kinetics possible. They are acknowledged in detail in the corresponding modules.

**License:** This educational material is available under an open-source license. Please cite this module when using it for teaching or research purposes.

[Back to top](#top)
